# Stock Analysis Runner (Google Colab)

Runs the four "major" NSE/BSE stock-analysis scripts from the [BMS_analysis_battery](https://github.com/herrrickshaw/BMS_analysis_battery) repo on Colab's compute instead of the local machine:

| Script | What it does |
|---|---|
| `nse_bse_extractor.py` | Bulk NSE/BSE data extraction for an index universe |
| `stock_metrics_nse.py` | Per-symbol news, PEG, PE×PB, RSI |
| `portfolio_analysis.py` | Portfolio beta, potential upside, efficient frontier (MPT) |
| `nse_bse_benchmark.py` | NSE vs BSE extraction-speed benchmark |

Each section clones/pulls the repo, installs only the lightweight `requirements_nse_bse.txt` dependencies, and shells out to the real scripts via `subprocess` — so this notebook stays a thin runner and the actual analysis logic keeps living (and getting fixed) in the repo, not duplicated here.

**Runtime:** CPU-only is fine — these scripts are I/O-bound (yfinance/NSE calls), not GPU-bound.

## 1. Clone the repo and install dependencies

In [ ]:
import os

REPO_URL = "https://github.com/herrrickshaw/BMS_analysis_battery.git"
REPO_DIR = "/content/BMS_analysis_battery"

if os.path.isdir(REPO_DIR):
    print("Repo already present, pulling latest...")
    !cd {REPO_DIR} && git pull -q
else:
    print("Cloning repo...")
    !git clone -q {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install -q -r requirements_nse_bse.txt
print("\nReady.")

## 2. Configuration

Edit these values, then run the section(s) you want below — sections are independent of each other.

In [ ]:
import subprocess, sys, json
import pandas as pd
from pathlib import Path

REPO = Path(REPO_DIR)

def run(cmd, **kw):
    """Run a repo script, streaming output, and return the CompletedProcess."""
    print("$ " + " ".join(cmd))
    proc = subprocess.run(cmd, cwd=REPO, text=True, capture_output=True, **kw)
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr, file=sys.stderr)
    return proc

# --- nse_bse_extractor.py ---
EXTRACTOR_EXCHANGE = "BOTH"       # NSE | BSE | BOTH
EXTRACTOR_INDEX = "NIFTY50"       # NIFTY50 for a quick run, NIFTY500 for the full universe
EXTRACTOR_MAX_SYMBOLS = 25        # cap for a quick Colab test run; set to None for the full index

# --- stock_metrics_nse.py ---
METRICS_SYMBOL = "RELIANCE"
METRICS_BATCH = ["RELIANCE", "TCS", "INFY"]   # used by the batch cell only

# --- portfolio_analysis.py ---
PORTFOLIO_SYMBOLS = ["RELIANCE", "TCS", "INFY", "HDFCBANK", "WIPRO"]
PORTFOLIO_WEIGHTS = None   # e.g. [0.3, 0.3, 0.2, 0.1, 0.1]; None = equal-weight

# --- nse_bse_benchmark.py ---
BENCHMARK_SYMBOLS = ["RELIANCE", "TCS"]

print("Configuration loaded.")

## 3. NSE/BSE bulk extractor

In [ ]:
cmd = [
    sys.executable, "nse_bse_extractor.py",
    "--exchange", EXTRACTOR_EXCHANGE,
    "--index", EXTRACTOR_INDEX,
    "--output-dir", "data",
]
if EXTRACTOR_MAX_SYMBOLS is not None:
    cmd += ["--max-symbols", str(EXTRACTOR_MAX_SYMBOLS)]

run(cmd)

# Show whatever CSV(s) landed in data/ most recently
csvs = sorted((REPO / "data").glob("*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if csvs:
    print(f"\nMost recent output: {csvs[0].name}")
    display(pd.read_csv(csvs[0]).head(20))
else:
    print("No CSV output found under data/ — check the log above.")

## 4. Per-symbol stock metrics (news, PEG, PE×PB, RSI)

In [ ]:
run([sys.executable, "stock_metrics_nse.py", METRICS_SYMBOL, "--rsi-period", "14", "--news", "5"])

In [ ]:
# Batch mode across several symbols
run([sys.executable, "stock_metrics_nse.py", "--batch", *METRICS_BATCH])

## 5. Portfolio beta / efficient frontier (MPT)

In [ ]:
from IPython.display import Image, display as ipy_display

cmd = [sys.executable, "portfolio_analysis.py", "--symbols", *PORTFOLIO_SYMBOLS, "--output", "efficient_frontier.png"]
if PORTFOLIO_WEIGHTS:
    cmd += ["--weights", *[str(w) for w in PORTFOLIO_WEIGHTS]]

run(cmd)

frontier_png = REPO / "efficient_frontier.png"
if frontier_png.exists():
    ipy_display(Image(filename=str(frontier_png)))
else:
    print("efficient_frontier.png not found — check the log above (--no-plot may have been used).")

## 6. NSE vs BSE extraction benchmark

In [ ]:
run([sys.executable, "nse_bse_benchmark.py", "--live", *BENCHMARK_SYMBOLS])

bench_csv = REPO / "reports" / "benchmark_results.csv"
if bench_csv.exists():
    df = pd.read_csv(bench_csv)
    display(df)
    df.plot(kind="bar", x=df.columns[0], title="NSE vs BSE extraction benchmark")
else:
    print("reports/benchmark_results.csv not found — check the log above.")

## 7. (Optional) Save generated reports to Google Drive

In [ ]:
SAVE_TO_DRIVE = False  # flip to True to persist outputs across Colab sessions

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive/stock_analysis_outputs")
    dest.mkdir(parents=True, exist_ok=True)
    import shutil
    for src in [REPO / "data", REPO / "reports", REPO / "efficient_frontier.png"]:
        if src.is_dir():
            shutil.copytree(src, dest / src.name, dirs_exist_ok=True)
        elif src.exists():
            shutil.copy(src, dest / src.name)
    print(f"Copied outputs to {dest}")
else:
    print("SAVE_TO_DRIVE is False — outputs only exist in this Colab session's /content.")